# 01 — Data QC and Preprocessing

This notebook verifies the data pipeline for EEG stimulus classification on the selected OpenNeuro ERP/oddball dataset.

## Goals

This notebook is used to confirm that:

1. the selected subjects load correctly
2. the chronological reconstruction of standard and target trials is correct
3. class counts and target rates are reasonable
4. contiguous chunk groups are created correctly for grouped evaluation
5. the selected crop window is applied correctly
6. control variants such as permuted labels and pre-stimulus-only cropping behave as expected at the data level

## Scope

This notebook does **not** train models.
Its purpose is to validate the input pipeline before moving to benchmarking.

## Expected outputs

All outputs are saved under the structured experiment folder:

`results/<dataset>/<task>/<signal>/<eval_mode>/<control_mode>/`

Expected outputs include:

- subject-level QC tables
- full-vs-cropped data comparison tables
- label distribution tables
- class-balance figures
- example trial figures
- experiment metadata JSON files

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment_family
from oddball_eeg.io import save_dataframe, save_experiment_metadata
from oddball_eeg.loader import (
    load_subjects_grouped,
    make_subject_summary_table,
    print_loaded_dataset_summary,
)
from oddball_eeg.preprocessing import (
    apply_crop_to_loaded_dataset,
    apply_permuted_labels_to_loaded_dataset,
    make_subject_label_table,
    print_crop_effect,
)
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)

In [ ]:
set_pandas_display_options()

## 1. Load experiment configuration

We start from the main within-subject benchmark configuration, because this will be the reference pipeline for the rest of the thesis workflow.

In [ ]:
# Change only this line when switching datasets.
EXPERIMENT_NAME = "ds003061_within_subject_max30_wide_timebin"
# EXPERIMENT_NAME = "ds005863_within_subject_max30_wide_timebin"
# EXPERIMENT_NAME = "ds006466_within_subject_max30_wide_timebin"
# EXPERIMENT_NAME = "ds006480_within_subject_max30_wide_timebin"

experiment_family = get_experiment_family(EXPERIMENT_NAME)

cfg = experiment_family["main"]
cfg_perm = experiment_family["permuted"]
cfg_prestim = experiment_family["prestim"]
cfg_loso = experiment_family["loso"]

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "01_data_qc_and_preprocessing.ipynb",
        "purpose": "Dataset loading, epoching, cropping, and data-level QC.",
        "experiment_key": EXPERIMENT_NAME,
    },
)

print("Experiment key :", EXPERIMENT_NAME)
print("Experiment name:", cfg.name)
print("Eval mode      :", cfg.eval_mode)
print("Control mode   :", cfg.control_mode)
print("Dataset        :", cfg.dataset.dataset)
print("Task           :", cfg.dataset.task)
print("Subjects       :", cfg.subjects.subjects)
print("Seeds          :", cfg.seeds.seeds)
print("Signal config  :", cfg.signal)
print("Crop window    :", cfg.effective_crop_window())
print("Chunk size     :", cfg.splits.group_chunk_size)
print("Purge neighbors:", cfg.splits.purge_neighbor_groups)

print("\nRelated experiment keys:")
print("main     :", EXPERIMENT_NAME)
print("permuted :", cfg_perm.result_dir())
print("prestim  :", cfg_prestim.result_dir())
print("loso     :", cfg_loso.result_dir())

print("\nResult directory:")
print(cfg.result_dir())

print("\nQC table directory:")
print(cfg.table_dir("qc"))

print("\nQC figure directory:")
print(cfg.figure_dir("qc"))

## 2. Load grouped chronological subject data

This step loads the selected subjects, reconstructs chronological trial order, and assigns contiguous chunk groups for grouped evaluation.

In [ ]:
epoching_cfg = make_epoching_config_from_experiment(cfg)

print(epoching_cfg)

data_full = load_subjects_grouped(
    dataset=cfg.dataset.dataset,
    task_name=cfg.dataset.task,
    subjects=cfg.subjects.subjects,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

In [ ]:
print_loaded_dataset_summary(data_full)

## 3. Subject-level summary table

This table gives a compact overview of the currently loaded dataset before any crop window is applied.

In [ ]:
subject_summary_full = make_subject_summary_table(data_full)
subject_summary_full = round_numeric_columns(subject_summary_full, decimals=4)

display(subject_summary_full)

save_dataframe(
    subject_summary_full,
    cfg.table_dir("qc") / "subject_summary_full_uncropped.csv",
)

## 4. Apply the main benchmark crop window

The main benchmark currently uses the post-stimulus signal window selected during development.

In [ ]:
crop_window = cfg.effective_crop_window()
assert crop_window is not None, "Expected an active crop window for the main benchmark."

data_main = apply_crop_to_loaded_dataset(
    data_full,
    tmin=crop_window[0],
    tmax=crop_window[1],
)

In [ ]:
print_crop_effect(data_full.times, data_main.times)

In [ ]:
subject_summary_main = make_subject_summary_table(data_main)
subject_summary_main = round_numeric_columns(subject_summary_main, decimals=4)

display(subject_summary_main)

save_dataframe(
    subject_summary_main,
    cfg.table_dir("qc") / "subject_summary_main_cropped.csv",
)

## 5. Compare the full loaded data vs the cropped main benchmark data

Cropping should reduce only the time dimension.
It should not change:
- number of trials
- class counts
- chunk grouping
- subject identity

In [ ]:
comparison_rows = []

for sub in data_full.subjects:
    comparison_rows.append({
        "subject": sub,
        "full_n_trials": int(data_full.X_by_sub[sub].shape[0]),
        "main_n_trials": int(data_main.X_by_sub[sub].shape[0]),
        "full_n_channels": int(data_full.X_by_sub[sub].shape[1]),
        "main_n_channels": int(data_main.X_by_sub[sub].shape[1]),
        "full_n_times": int(data_full.X_by_sub[sub].shape[2]),
        "main_n_times": int(data_main.X_by_sub[sub].shape[2]),
        "full_n_target": int((data_full.y_by_sub[sub] == 1).sum()),
        "main_n_target": int((data_main.y_by_sub[sub] == 1).sum()),
        "full_n_groups": int(pd.Series(data_full.groups_by_sub[sub]).nunique()),
        "main_n_groups": int(pd.Series(data_main.groups_by_sub[sub]).nunique()),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("subject").reset_index(drop=True)
comparison_df = round_numeric_columns(comparison_df, decimals=4)

display(comparison_df)

save_dataframe(
    comparison_df,
    cfg.table_dir("qc") / "full_vs_main_crop_comparison.csv",
)

## 6. Quick label table for the main benchmark data

In [ ]:
label_table_main = pd.DataFrame(
    make_subject_label_table(data_main),
    columns=["subject", "n_standard", "n_target", "target_rate"],
)

label_table_main = round_numeric_columns(label_table_main, decimals=4)

display(label_table_main)

save_dataframe(
    label_table_main,
    cfg.table_dir("qc") / "label_table_main_cropped.csv",
)

## 7. Visual inspection of example trials

We plot a few standard and target trials from one representative electrode (`Pz` if available) to confirm that the data looks reasonable after loading and cropping.

In [ ]:
subject_to_plot = data_main.subjects[0]
X_plot = data_main.X_by_sub[subject_to_plot]
y_plot = data_main.y_by_sub[subject_to_plot]
times_plot = data_main.times
ch_names_plot = data_main.ch_names

preferred_channels = ["Pz", "CP1", "CP2", "C3", "C4"]
plot_ch = next((ch for ch in preferred_channels if ch in ch_names_plot), ch_names_plot[0])
plot_ch_idx = ch_names_plot.index(plot_ch)

std_idx = np.where(y_plot == 0)[0][:5]
tgt_idx = np.where(y_plot == 1)[0][:5]

print("Subject:", subject_to_plot)
print("Plot channel:", plot_ch)
print("Standard trial indices:", std_idx)
print("Target trial indices:", tgt_idx)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for idx in std_idx:
    ax.plot(times_plot, X_plot[idx, plot_ch_idx], alpha=0.7)

ax.axvline(0.0, linestyle="--")
ax.set_title(
    f"{cfg.dataset.dataset} | Subject {subject_to_plot} — first 5 standard trials at {plot_ch}"
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.grid(True, alpha=0.3)

fig.tight_layout()

save_figure(
    fig,
    cfg.figure_dir("qc", "example_trials") / f"subject-{subject_to_plot}_standard_trials_{plot_ch}.png",
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for idx in tgt_idx:
    ax.plot(times_plot, X_plot[idx, plot_ch_idx], alpha=0.7)

ax.axvline(0.0, linestyle="--")
ax.set_title(f"Subject {subject_to_plot} — first 5 target trials at {plot_ch}")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.grid(True, alpha=0.3)

fig.tight_layout()

save_figure(
    fig,
    cfg.figure_dir("qc", "example_trials") / f"subject-{subject_to_plot}_target_trials_{plot_ch}.png",
)

plt.show()

In [ ]:
roi_candidates = ["Pz", "CP1", "CP2", "CP5", "CP6", "C3", "C4"]
roi_idx = [ch_names_plot.index(ch) for ch in roi_candidates if ch in ch_names_plot]

if len(roi_idx) == 0:
    roi_idx = [plot_ch_idx]

erp_std = X_plot[y_plot == 0][:, roi_idx, :].mean(axis=(0, 1))
erp_tgt = X_plot[y_plot == 1][:, roi_idx, :].mean(axis=(0, 1))

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(times_plot, erp_std, label="standard")
ax.plot(times_plot, erp_tgt, label="target")
ax.axvline(0.0, linestyle="--", linewidth=1)
ax.axvspan(0.25, 0.50, alpha=0.15, label="P300 window")

ax.set_title(f"ERP sanity check — subject {subject_to_plot}, ROI average")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()

save_figure(
    fig,
    cfg.figure_dir("qc", "erp") / f"subject-{subject_to_plot}_erp_sanity_roi.png",
)

plt.show()

## 8. Plot subject-wise class balance

This helps confirm that target-vs-standard imbalance varies by subject and should be considered during training and evaluation.

In [ ]:
plot_df = subject_summary_main.copy()

x = np.arange(len(plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, plot_df["n_standard"], width=width, label="Standard")
ax.bar(x + width / 2, plot_df["n_target"], width=width, label="Target")

ax.set_xticks(x)
ax.set_xticklabels(plot_df["subject"].astype(str).tolist())
ax.set_xlabel("Subject")
ax.set_ylabel("Number of trials")
ax.set_title(
    f"Class balance by subject — {cfg.dataset.dataset}, {cfg.dataset.task}"
)
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()

save_figure(
    fig,
    cfg.figure_dir("qc", "class_balance") / "class_balance_by_subject_main_cropped.png",
)

plt.show()

## 9. Build the pre-stimulus-only control dataset

This dataset uses only the pre-stimulus window and will later be used as an anti-overfitting control.

In [ ]:
save_experiment_metadata(
    cfg_prestim,
    extra={
        "notebook": "01_data_qc_and_preprocessing.ipynb",
        "purpose": "Pre-stimulus control data-level QC.",
        "main_experiment_key": EXPERIMENT_NAME,
    },
)

prestim_window = cfg_prestim.effective_crop_window()
assert prestim_window is not None

print("Pre-stim result directory:")
print(cfg_prestim.result_dir())

data_prestim = apply_crop_to_loaded_dataset(
    data_full,
    tmin=prestim_window[0],
    tmax=prestim_window[1],
)

In [ ]:
print_crop_effect(data_full.times, data_prestim.times)

In [ ]:
subject_summary_prestim = make_subject_summary_table(data_prestim)
subject_summary_prestim = round_numeric_columns(subject_summary_prestim, decimals=4)

display(subject_summary_prestim)

save_dataframe(
    subject_summary_prestim,
    cfg_prestim.table_dir("qc") / "subject_summary_prestim_cropped.csv",
)

## 10. Build the permuted-label control dataset

This keeps the same data and class counts per subject, but destroys the mapping between trials and labels.

In [ ]:
save_experiment_metadata(
    cfg_perm,
    extra={
        "notebook": "01_data_qc_and_preprocessing.ipynb",
        "purpose": "Permuted-label control data-level QC.",
        "main_experiment_key": EXPERIMENT_NAME,
        "permutation_seed": 12345,
    },
)

data_perm = apply_permuted_labels_to_loaded_dataset(
    data_main,
    seed=12345,
)

print("Permuted-label result directory:")
print(cfg_perm.result_dir())

In [ ]:
perm_rows = []

for sub in data_main.subjects:
    y_main = data_main.y_by_sub[sub]
    y_perm = data_perm.y_by_sub[sub]

    perm_rows.append({
        "subject": sub,
        "main_n_standard": int((y_main == 0).sum()),
        "main_n_target": int((y_main == 1).sum()),
        "perm_n_standard": int((y_perm == 0).sum()),
        "perm_n_target": int((y_perm == 1).sum()),
        "same_label_fraction": float(np.mean(y_main == y_perm)),
    })

perm_check_df = pd.DataFrame(perm_rows).sort_values("subject").reset_index(drop=True)
perm_check_df = round_numeric_columns(perm_check_df, decimals=4)

display(perm_check_df)

save_dataframe(
    perm_check_df,
    cfg_perm.table_dir("qc") / "permuted_label_check.csv",
)

## 11. QC conclusions

At this point we want to confirm the following before moving to benchmarking:

- subjects load successfully
- chronological trial streams are constructed
- grouped chunking is present
- cropping changes only the time dimension
- pre-stimulus-only data is available
- permuted labels preserve class counts but break trial-label correspondence

If these checks look correct, the data pipeline is ready for the benchmarking notebook.

In [ ]:
dataset_qc_summary = {
    "dataset": cfg.dataset.dataset,
    "task": cfg.dataset.task,
    "subjects": list(data_main.subjects),
    "n_subjects": int(len(data_main.subjects)),
    "n_channels": int(len(data_main.ch_names)),
    "n_times_full": int(len(data_full.times)),
    "n_times_main": int(len(data_main.times)),
    "n_times_prestim": int(len(data_prestim.times)),
    "sfreq": float(data_main.sfreq),
    "full_time_min": float(data_full.times[0]),
    "full_time_max": float(data_full.times[-1]),
    "main_time_min": float(data_main.times[0]),
    "main_time_max": float(data_main.times[-1]),
    "prestim_time_min": float(data_prestim.times[0]),
    "prestim_time_max": float(data_prestim.times[-1]),
    "group_chunk_size": int(cfg.splits.group_chunk_size),
    "n_splits": int(cfg.splits.n_splits),
    "purge_neighbor_groups": int(cfg.splits.purge_neighbor_groups),
}

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "01_data_qc_and_preprocessing.ipynb",
        "status": "completed",
        "dataset_qc_summary": dataset_qc_summary,
    },
)

dataset_qc_summary